# AI Control Queue — Analysis - Stage 1

This notebook demonstrates a minimal implementation of a stateful, observable queue
for AI-enabled workflows.

Goal:
- validate queue behavior
- demonstrate deterministic state control
- show traceability via events
- measure basic system properties

Core principle:
Non-deterministic AI components require deterministic control boundaries.

State Machine:
INBOUND → WIP → RESOLVED

# BLOCK 1 — DB + Basic Operations

## 1. DB Connection

In [2]:
import psycopg
import uuid
import json
from datetime import datetime
conn = psycopg.connect(
    "dbname=ai_queue user=ai_queue_user password=ai_queue_password host=localhost port=5432"
)
conn.autocommit = False

## 2. create_item()

In [3]:
def create_item(source, raw_data, state="INBOUND"):
    with conn.cursor() as cur:
        cur.execute(
            """
            INSERT INTO queue_items (id, source, raw_data, state)
            VALUES (%s, %s, %s, %s)
            RETURNING id;
            """,
            (uuid.uuid4(), source, json.dumps(raw_data), state),
        )
        item_id = cur.fetchone()[0]
        conn.commit()
        return item_id

## 3. get_items_by_state()

In [4]:
def get_items_by_state(state):
    with conn.cursor() as cur:
        cur.execute(
            "SELECT id, state, created_at FROM queue_items WHERE state = %s;", (state,),
        )
        return cur.fetchall()

# BLOCK 2 — State Control

## 4. transition_item()

This is the core function — must be correct.

In [8]:
def transition_item(item_id, next_state, actor="system", reason=None):
    with conn.cursor() as cur:
        # get current state
        cur.execute(
            "SELECT state FROM queue_items WHERE id = %s FOR UPDATE;",
            (item_id,),
        )
        row = cur.fetchone()
        if not row:
            raise Exception("Item not found")

        prev_state = row[0]

        # update item
        cur.execute(
            """
            UPDATE queue_items
            SET state = %s, updated_at = NOW()
            WHERE id = %s;
            """,
            (next_state, item_id),
        )

        # append event
        cur.execute(
            """
            INSERT INTO queue_events (event_id, item_id, previous_state, next_state, actor, reason)
            VALUES (%s, %s, %s, %s, %s, %s);
            """,
            (uuid.uuid4(), item_id, prev_state, next_state, actor, reason),
        )

        conn.commit()

## 5. get_item_events()

In [9]:
def get_item_events(item_id):
    with conn.cursor() as cur:
        cur.execute(
            """
            SELECT previous_state, next_state, actor, created_at
            FROM queue_events
            WHERE item_id = %s
            ORDER BY created_at;
            """,
            (item_id,),
        )
        return cur.fetchall()

# BLOCK 3 — Behavior / Observability

### 6. detect_stuck_items()

In [10]:
def detect_stuck_items(threshold_minutes=5):
    with conn.cursor() as cur:
        cur.execute(
            """
            SELECT id, state, NOW() - updated_at AS time_in_state
            FROM queue_items
            WHERE NOW() - updated_at > INTERVAL '%s minutes';
            """ % threshold_minutes
        )
        return cur.fetchall()

## 7. Small Demo Run

In [35]:
# Create an item
item_id = create_item(
    "email",
    {"from": "recruiter@example.com", "subject": "AI Role"}
)

In [36]:
def move_items_through_queue():
    # Move items through the states: INBOUND > WIP
    items = get_items_by_state('INBOUND')
    print(f'Found {len(items)} INBOUND items')
    print(items)
    for item in items:
        transition_item(item_id, "WIP")
    # Move item through the states: WIP > RESOLVED
    items = get_items_by_state('WIP')
    print(items)
    print(f'Found {len(items)} WIP items')
    for item in items:
        transition_item(item_id, "RESOLVED")

In [ ]:
def print_item_by_state(state):
    resolved_items = get_items_by_state(state)
    for item in resolved_items:
        item_id = item[0]
        print(f'{item[2].strftime("%Y%m%d %H:%M")}: item_id = {item_id}, current_state = {item[1]}')
        item_events = get_item_events(item_id)
        for event in item_events:
            print(f'> {event[3].strftime("%Y%m%d %H:%M")}: owner={event[2]} | state update = {event[0]} > {event[1]}')

In [38]:
move_items_through_queue()

Found 1 INBOUND items
[(UUID('f5799ed1-6d9c-4acf-8dcd-b5e45e83743d'), 'INBOUND', datetime.datetime(2026, 5, 3, 1, 15, 16, 176661, tzinfo=zoneinfo.ZoneInfo(key='Etc/UTC')))]
[(UUID('f5799ed1-6d9c-4acf-8dcd-b5e45e83743d'), 'WIP', datetime.datetime(2026, 5, 3, 1, 15, 16, 176661, tzinfo=zoneinfo.ZoneInfo(key='Etc/UTC')))]
Found 1 WIP items


In [41]:
print('INBOUND items:')
print_item_by_state("INBOUND")

print('WIP items:')
print_item_by_state("WIP")

print('Resolved items:')
print_item_by_state("RESOLVED")

INBOUND items:
WIP items:
Resolved items:
20260503 00:25: item_id = 7aba6c5e-5ece-45da-9ab3-ea197eb33e1b, current_state = RESOLVED
> 20260503 00:25: owner=system | state update = INBOUND > WIP
> 20260503 00:25: owner=system | state update = WIP > RESOLVED
20260503 00:25: item_id = 6f657b19-a9ca-4885-928c-c4348f2cf62c, current_state = RESOLVED
> 20260503 00:25: owner=system | state update = INBOUND > WIP
> 20260503 00:25: owner=system | state update = WIP > RESOLVED
> 20260503 00:25: owner=system | state update = RESOLVED > WIP
> 20260503 00:49: owner=system | state update = WIP > RESOLVED
> 20260503 00:49: owner=system | state update = RESOLVED > WIP
> 20260503 01:02: owner=system | state update = WIP > RESOLVED
20260502 23:12: item_id = 0ad67fb6-71d5-476d-863d-af44be963bab, current_state = RESOLVED
> 20260503 01:02: owner=system | state update = INBOUND > WIP
> 20260503 01:07: owner=system | state update = WIP > RESOLVED
20260503 01:13: item_id = 1894f00f-a8ee-4350-b3dd-06ad1bd23b32, 

# BLOCK 4 — Metrics

## 8. Basic Metrics

In [31]:
def count_by_state():
    with conn.cursor() as cur:
        cur.execute(
            """
            SELECT state, COUNT(*)
            FROM queue_items
            GROUP BY state;
            """
        )
        return cur.fetchall()

In [32]:
def avg_time_in_state():
    with conn.cursor() as cur:
        cur.execute(
            """
            SELECT state, AVG(NOW() - updated_at)
            FROM queue_items
            GROUP BY state;
            """
        )
        return cur.fetchall()

In [33]:
count_by_state()

[('RESOLVED', 4)]

In [34]:
avg_time_in_state()

[('RESOLVED', datetime.timedelta(seconds=1062, microseconds=362096))]

# BLOCK Summary

## 9. Important Engineering Notes
### 1. Transaction boundary
transition_item() must:
read state
update state
insert event
commit once
### 2. State ownership
only transition_item() modifies state
nothing else touches queue_items.state
### 3. No LLM yet
keep deterministic
LLM comes later (optional layer)

### 4. Minimalism
This instrument does NOT include:
classes
abstractions
frameworks

### What This Notebook Proves

In high load, low observability operational environments, after execution, there is:

* working stateful queue
* event-based history
* observable lifecycle
* measurable system behavior

Which directly validates:

* “The queue decomposes uncertainty into observable units”

## 10. Summary and Future follow ups: Transition Policy and Abnormality Detection
As demonstrated, one of the items stuck in the queue and flipped between RESOLVED and WIP states multiple times.

Further enhancements recommended:
* is_allowed_transition(previous_state, next_state)
* detect_terminal_reopens()
* detect_repeated_transitions()
* detect_items_with_many_transitions()

The queue’s purpose is not just to “hold work.” It enforces **processing order at the boundary**:

```text
INBOUND → WIP → RESOLVED
```

So:

```text
RESOLVED → WIP
```

should **not** be treated as normal flow.

It should be treated as a **rework / correction / recovery event**.

My recommendation:

### Keep the primary state simple

```text
INBOUND → WIP → RESOLVED
```

Do **not** add `REOPENED` as a normal state yet.

Instead, keep:

```text
RESOLVED → WIP
```

allowed only as an **exception transition** with required metadata:

```text
transition_type = "reopen"
reason = required
actor = "human" | "recovery_policy"
```

This preserves operational simplicity:

```sql
SELECT * FROM queue_items WHERE state = 'WIP';
```

while still allowing observability through events.

### Add abnormality metrics

This is where the value appears:

```sql
SELECT item_id, COUNT(*) AS transition_count
FROM queue_events
GROUP BY item_id
ORDER BY transition_count DESC;
```

Detect resolved items reopened:

```sql
SELECT item_id, COUNT(*) AS reopen_count
FROM queue_events
WHERE previous_state = 'RESOLVED'
  AND next_state = 'WIP'
GROUP BY item_id;
```

Detect flip-flopping:

```sql
SELECT item_id, COUNT(*) AS resolved_to_wip_count
FROM queue_events
WHERE previous_state = 'RESOLVED'
  AND next_state = 'WIP'
GROUP BY item_id
HAVING COUNT(*) > 1;
```

### Design conclusion

`REOPENED` should probably remain an **event classification**, not a primary queue state — at least for v0.1.

Because the queue exists to enforce simple lifecycle order. Reopen is not a lifecycle stage; it is an **exception to lifecycle completion**.

Clean distinction:

```text
state = current operational position
event_type = why/how the transition happened
```

So:

```text
queue_items.state = WIP
queue_events.event_type = reopen
```

That gives both:

* simple work selection
* visible abnormality history

Best formulation for the paper / analysis:

> Terminal-state reversals are allowed only as explicit recovery events. They do not become normal workflow stages unless their frequency justifies promoting them into the state machine.

That last sentence is the key design principle.
